In [0]:
import boto3
import csv
import json
import io
import random
from datetime import datetime, timezone, date


### define env vars

In [0]:
# Accessing data directly through Spark/DBFS
BUCKET = 's3://fraud-demo-bucket-043309328060-us-east-1-an/'

### define exps 

In [0]:

COUNTRIES   = ["GB","US","EU","DE","FR","JP","CN","AU","CA"]
ENTITY_TYPES = ["INDIVIDUAL","ORGANISATION","VESSEL","AIRCRAFT"]
SANCTIONS_PROGRAMS = [
    "OFAC_SDN","UN_CONSOLIDATED","EU_CONSOLIDATED",
    "HMT_CONSOLIDATED","OFAC_CONSOLIDATED"
]


### define functions

In [0]:

def generate_sanctions_json(n=200):
    records = []
    for i in range(n):
        records.append({
            "entity_id":        f"ENT{str(i+1).zfill(8)}",
            "entity_type":      random.choice(ENTITY_TYPES),
            "full_name":        f"Sanctioned Entity {i+1}",
            "aliases":          [f"Alias {i+1}A", f"Alias {i+1}B"],
            "country_of_origin":random.choice(COUNTRIES),
            "sanctions_program":random.choice(SANCTIONS_PROGRAMS),
            "listed_date":      "2023-06-15",
            "last_updated":     date.today().isoformat(),
            "is_active":        random.choice([True,True,True,False]),
            "risk_score":       round(random.uniform(0.5, 1.0), 3),
            "source_list":      random.choice(SANCTIONS_PROGRAMS),
        })
    return json.dumps({
        "schema_version": "1.0",
        "generated_at":   datetime.now(timezone.utc).isoformat(),
        "record_count":   n,
        "records":        records
    }, indent=2).encode("utf-8")


In [0]:


data = generate_sanctions_json()

# Writing as CSV
s3 = boto3.client('s3')

bucket_name = "fraud-demo-bucket-043309328060-us-east-1-an"
file_key = f"reference/sanctions/{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}.json"
# Write the object
s3.put_object(
    Bucket=bucket_name,
    Key=file_key,
    Body=json.dumps(data).encode('utf-8'),
    ContentType='application/json'
)